# nano-jev smoke test (Kaggle)

Installs [`nano-jev`](https://pypi.org/project/nano-jev/) from PyPI, downloads the v0.1 weights from
[Hugging Face](https://huggingface.co/sdmlai/nano-jev) and checks that every feature works.

**Before running:** in the notebook sidebar, *Settings → Internet → On*. A GPU is optional
(the model is 22.7M parameters and runs fine on CPU).

Then **Run All**. The last cell prints `ALL SMOKE TESTS PASSED`.

In [ ]:
# Install from PyPI.
!pip install -q nano-jev

# Before the PyPI release (or to test the latest code), install from GitHub instead:
# !pip install -q git+https://github.com/shubham10divakar/nano-jev

## 1. Versions and hardware

In [ ]:
import nanojev, torch, transformers
print("nano-jev     ", nanojev.__version__, "| default weights", nanojev.DEFAULT_VERSION)
print("torch        ", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("transformers ", transformers.__version__)

## 2. Command-line tool: list the available weights

In [ ]:
!nano-jev --version
!nano-jev list

## 3. Download and load the weights

In [ ]:
import time
t0 = time.time()
d = nanojev.load("v0.1")          # downloads ~90 MB on first use, then cached
print(f"loaded version {d.version} on {d.device} in {time.time() - t0:.1f}s")
print("calibration temperatures:", d.temperatures)
assert d.version == "0.1"

## 4. Decisions

In [ ]:
# Relevance: 3 levels per passage
rel = d.relevance("When was UCL founded?", [
    "University College London was founded in 1826.",
    "The Analytical Engine was a proposed mechanical computer.",
])
for r in rel:
    print({k: round(v, 3) for k, v in r.items()})
assert max(rel[0], key=rel[0].get) == "directly answers"
assert max(rel[1], key=rel[1].get) == "irrelevant"

In [ ]:
# Sufficiency: a two-hop question needs both passages
q = "In which year was the university attended by Ada Lovelace's tutor founded?"
both = ["Augustus De Morgan tutored Ada Lovelace. He was a professor at University College London.",
        "University College London was founded in 1826."]
full, partial = d.sufficient(q, both), d.sufficient(q, both[:1])
print(f"both passages P(yes)={full['yes']:.3f} | first only P(yes)={partial['yes']:.3f}")
assert full["yes"] > partial["yes"]

In [ ]:
# Groundedness: is the claim supported by the context?
ctx = "University College London was founded in 1826."
true_, false_ = d.grounded("UCL was founded in 1826.", ctx), d.grounded("UCL was founded in 1900.", ctx)
print(f"true claim P(yes)={true_['yes']:.3f} | false claim P(yes)={false_['yes']:.3f}")
assert true_["yes"] > 0.5 > false_["yes"]

In [ ]:
# Custom options the model never saw in training
out = d.decide("Which topic is this passage about?", ["computing", "cooking", "football"],
               "The Analytical Engine was a proposed mechanical computer.")
print({k: round(v, 3) for k, v in out.items()})
assert abs(sum(out.values()) - 1) < 1e-5

## 5. Speed

In [ ]:
passages = ["University College London was founded in 1826."] * 256
t0 = time.time()
d.relevance("When was UCL founded?", passages)
print(f"{1000 * (time.time() - t0) / len(passages):.2f} ms per decision on {d.device}")

## 6. CLI decisions and the full scripted smoke test

In [ ]:
!nano-jev relevance --json -q "When was UCL founded?" -p "UCL was founded in 1826."
!nano-jev grounded --claim "UCL was founded in 1900." --context "University College London was founded in 1826."

In [ ]:
# The same checks as a script (exits non-zero on failure)
!wget -q https://raw.githubusercontent.com/shubham10divakar/nano-jev/main/examples/smoke_test.py -O smoke_test.py
!python smoke_test.py